# Atajos: qué podría aprender el modelo en lugar de la morfología

Fase 2 (MaNGA + Legacy Survey). Antes de entrenar con `.jpg` o `.fits` conviene saber qué
atributos, **ajenos a la forma de la galaxia**, permitirían acertar la etiqueta temprana/tardía.
Si existen, una red puede aprenderlos en vez de la morfología: el AUC sale alto en validación y el
modelo falla en cuanto cambia la distribución (otra región del cielo, otro rango de distancias).

Un atributo es un **atajo** solo si cumple dos condiciones:

1. **Predice la etiqueta** en los datos (AUC lejos de 0.5).
2. **Es visible en la imagen** (tamaño aparente, nitidez, color de render, ruido...).

Se distinguen dos tipos de atributo:

- **Observacionales** (no deberían informar la clase): redshift, región del survey, zoom del
  cutout, latitud galáctica.
- **Físicos** (correlatos legítimos de la morfología): masa estelar, tamaño en kpc. No son trampa,
  pero confunden: hay que saber qué parte de la señal de un atributo observacional pasa por ellos.

El análisis usa el **catálogo curado completo** (`MORFOLOGIA_CURATED_CSV`, sin imágenes) para la
condición 1 y la muestra de 50 `.fits` para una primera mirada a la condición 2. La lógica vive en
`src/data/shortcuts.py`; ver también `README.md`, sección "Atajos: qué podría aprender el modelo en
lugar de la morfología".

In [ ]:
# importaciones de sistema
import sys
from pathlib import Path

# Referencia a otros archivos sin necesidad de instalar el paquete
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# importaciones de librerias de terceros
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# configuracion de proyecto
from src.config import (
    CUTOUTS_INDEX_CSV,
    EARLY_TYPE_LABEL,
    LATE_TYPE_LABEL,
    LEGACY_BANDS,
    LEGACY_LAYER,
    LEGACY_REGION_NORTH,
    LEGACY_REGION_SOUTH,
    MORFOLOGIA_CURATED_CSV,
    RANDOM_SEED,
    SHORTCUT_LOGREG_C,
    SHORTCUT_MATCH_N_BINS,
)
from src.data.fits_photometry import build_fits_photometry_table
from src.data.shortcuts import (
    OBSERVATIONAL_COLUMNS,
    PHYSICAL_COLUMNS,
    auc_table,
    galaxy_groups,
    label_to_int,
    matching_weights,
    observational_attributes,
    weighted_auc,
)
from src.evaluate import predictability_test
from src.utils import set_seed

set_seed(RANDOM_SEED)

# Colores fijos por entidad (los mismos que notebook/exploration_fits.ipynb).
LABEL_COLORS = {EARLY_TYPE_LABEL: "#eb6834", LATE_TYPE_LABEL: "#2a78d6"}
KIND_COLORS = {"observacional": "#4a3aa7", "físico": "#1baf7a"}
plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e5e4e0",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8984",
    "figure.dpi": 100,
})

## 1. Datos: catálogo curado y atributos de observación

`observational_attributes` añade a cada galaxia la región del survey, |latitud galáctica|, el zoom
con que se descarga su cutout (`compute_pixscale`, el mismo del descargador), si ese zoom es más
fino que el píxel nativo (`ampliada`: imagen interpolada), si quedó en un límite (`zoom_topado`) y
el tamaño físico `r90_kpc` (cosmología `COSMOLOGY` de `src/config.py`).

In [ ]:
catalog = observational_attributes(pd.read_csv(MORFOLOGIA_CURATED_CSV))
print(f"{len(catalog)} galaxias")
catalog["label"].value_counts().to_frame("galaxias").assign(fraccion=lambda t: (t["galaxias"] / t["galaxias"].sum()).round(3))

## 2. Panorama: ¿qué atributo predice la etiqueta por sí solo?

AUC de cada atributo usado solo como puntaje, con IC 95% por bootstrap (`auc_table`). AUC = 0.5 es
azar; la dirección dice hacia qué clase empujan los valores altos.

In [ ]:
overview = auc_table(catalog, OBSERVATIONAL_COLUMNS + PHYSICAL_COLUMNS)
overview["tipo"] = ["físico" if c in PHYSICAL_COLUMNS else "observacional" for c in overview.index]
overview.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ordered = overview.sort_values("auc")
for i, (name, row) in enumerate(ordered.iterrows()):
    color = KIND_COLORS[row["tipo"]]
    ax.plot([row["ic_inf"], row["ic_sup"]], [i, i], color=color, linewidth=2)
    ax.scatter(row["auc"], i, s=60, color=color, edgecolor="white", linewidth=1.5, zorder=3)
    ax.text(row["ic_sup"] + 0.005, i, f"{row['auc']:.2f}", va="center", fontsize=8, color="#52514e")
ax.set_yticks(range(len(ordered)), ordered.index)
ax.axvline(0.5, color="#8a8984", linewidth=1, linestyle="--")
ax.set_xlabel("AUC (orientado; 0.5 = azar)")
for kind, color in KIND_COLORS.items():
    ax.scatter([], [], color=color, label=kind)
ax.legend(frameon=False, loc="lower right")
ax.set_title("Cada atributo solo, catálogo curado completo", fontsize=10, loc="left")
plt.tight_layout()
plt.show()

Tres atributos se separan claramente del azar: **inclinación**, **masa** y **redshift**. Los
observacionales de la imagen (región, zoom, interpolación, latitud galáctica, tamaño aparente)
quedan entre 0.50 y 0.55. Cada uno se revisa abajo.

## 3. Redshift: ¿atajo propio o reflejo de la masa?

Las tempranas están en promedio más lejos. A primera vista es el atajo más peligroso: una galaxia
lejana se ve más chica, más tenue y más borrosa, y la red podría aprender "lejos = temprana".

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(0, 0.15, 46)
for label, group in catalog.groupby("label"):
    axes[0].hist(group["nsa_z"], bins=bins, density=True, histtype="step", linewidth=2,
                 color=LABEL_COLORS[label], label=f"{label} (mediana {group['nsa_z'].median():.3f})")
axes[0].set_xlabel("redshift (nsa_z)")
axes[0].set_ylabel("densidad")
axes[0].legend(frameon=False)
axes[0].set_title("Redshift por clase", fontsize=10, loc="left")

sample = catalog.sample(4000, random_state=RANDOM_SEED)
for label, group in sample.groupby("label"):
    axes[1].scatter(group["LogMass"], group["nsa_z"], s=6, alpha=0.5, color=LABEL_COLORS[label], label=label)
axes[1].set_xlabel("log10 masa estelar")
axes[1].set_ylabel("redshift (nsa_z)")
axes[1].legend(frameon=False, markerscale=3)
axes[1].set_title("Redshift vs masa (4,000 galaxias al azar)", fontsize=10, loc="left")
plt.tight_layout()
plt.show()

El panel derecho muestra la **selección de MaNGA**: la muestra se diseñó para que el IFU cubra la
galaxia hasta un radio fijo, así que a mayor masa (más luminosa, más grande) se eligen galaxias más
lejanas. Las bandas diagonales son las submuestras del survey. Y como las tempranas son más masivas,
también quedan más lejos.

Para separar los dos efectos se **reponderan las clases** (`matching_weights`) para que tengan la
misma distribución de una variable, y se mide el AUC que queda en las demás. Si el redshift solo
predice la clase a través de la masa, su AUC a masa igual debe caer a ~0.5.

In [ ]:
scenarios = {
    "sin ponderar": None,
    "a masa igual": ["LogMass"],
    "a redshift igual": ["nsa_z"],
    "a masa y redshift iguales": ["LogMass", "nsa_z"],
}
rows = {}
for scenario, on in scenarios.items():
    weights = pd.Series(1.0, index=catalog.index) if on is None else matching_weights(catalog, on)
    rows[scenario] = {
        column: weighted_auc(catalog, column, weights)
        for column in ["nsa_z", "LogMass", "r90_kpc", "incl", "PETRO_TH90", "sur"]
    }
    rows[scenario]["fuera de soporte"] = float((weights == 0).mean())
conditional = pd.DataFrame(rows).T
print(f"AUC sin orientar (> 0.5 = valores altos van con tardía); {SHORTCUT_MATCH_N_BINS} cuantiles por variable")
conditional.round(3)

- **A masa igual, el redshift no separa nada** (AUC ≈ 0.51). Toda su señal pasa por la masa: es
  un efecto de la selección de MaNGA, no una propiedad que distinga la morfología.
- Al revés no pasa: **a redshift igual, la masa sigue separando** (≈ 0.41, es decir 0.59
  orientado). La masa es el correlato físico real.
- La inclinación casi no cambia al igualar masa o redshift: es una señal independiente (sección 6).

**¿Es visible el redshift en la imagen?** El encuadre se pide como `4 × PETRO_TH90`, así que todas
las galaxias llegan con el mismo tamaño aparente en píxeles. Lo que delataría la distancia es la
**resolución**: una galaxia lejana ocupa menos arcsec y la PSF la suaviza más. Ese efecto lo mide
`PETRO_TH90` (tamaño angular), que solo da AUC 0.52. El atenuamiento de brillo superficial
`(1+z)^4` es de ~0.1–0.4 mag para z ≤ 0.1, y desaparece si las imágenes se normalizan por galaxia.

**Conclusión:** el redshift está correlacionado con la etiqueta (AUC 0.67), pero lo que la imagen
muestra del redshift casi no lo está. Riesgo **bajo a moderado**. Igual conviene verificarlo con el
modelo entrenado: si su AUC cae al evaluarlo con pesos `a redshift igual`, está usando la distancia.

## 4. Región del survey (norte / sur)

La región cambia el telescopio, los filtros y, en `ls-dr10_grz`, el render del visor (el morado del
sur; ver `notebook/exploration_fits.ipynb`). Para que sea un atajo, además tiene que predecir la
etiqueta:

In [ ]:
region_table = pd.crosstab(catalog["region"], catalog["label"], margins=True, margins_name="total")
region_table["fraccion_tardia"] = (region_table[LATE_TYPE_LABEL] / region_table["total"]).round(3)
region_table

La proporción de tardías es prácticamente igual en las dos regiones (AUC 0.51). **La región no es
un atajo de la etiqueta**: un modelo que aprenda "esto es del sur" no gana nada.

Lo que sí es, es un **cambio de dominio**. Si la red gasta capacidad en el color del render, puede
funcionar peor en una región que en la otra, o peor al pasar de una región a la otra. Eso se mide con
el modelo entrenado y no aquí: AUC por región y entrenando en una región y evaluando en la otra
(`cross_group_auc` en `src/evaluate.py`). Esto **corrige el matiz** de README.md, sección
"Variantes de capa y bandas": el riesgo del morado no es que la red aprenda la declinación como
atajo, es que generalice peor entre regiones.

## 5. Encuadre y zoom del cutout

El zoom de cada cutout depende de `PETRO_TH90`. Si una clase tuviera sistemáticamente imágenes más
interpoladas (`ampliada`) o con zoom en un límite (`zoom_topado`), la textura de la imagen podría
delatarla.

In [ ]:
reasons = pd.DataFrame({
    "sin PETRO_TH90": catalog["PETRO_TH90"].isna(),
    "zoom mínimo (galaxia chica)": np.isclose(catalog["pixscale"], catalog["pixscale"].min()) & catalog["PETRO_TH90"].notna(),
    "zoom máximo (galaxia grande)": np.isclose(catalog["pixscale"], catalog["pixscale"].max()),
    "ampliada (interpolada)": catalog["ampliada"].astype(bool),
})
summary = reasons.groupby(catalog["label"]).mean().T.round(3)
summary["n_total"] = reasons.sum()
summary

- **~70% de las imágenes son interpoladas** (el zoom pedido es más fino que el píxel nativo de
  0.262″), en proporciones casi iguales en las dos clases. No es un atajo, pero sí es una propiedad
  de todo el dataset que conviene recordar: en esas imágenes los "detalles" finos son interpolación,
  no información.
- `zoom_topado` llega a AUC 0.55 por las galaxias en los límites del zoom. Son pocas: se pueden
  marcar en el control de calidad y revisar si alguna clase está sobrerrepresentada.

## 6. Inclinación

`incl` es la señal individual más fuerte (AUC 0.72), y **la inclinación es un atributo
observacional**: la misma galaxia vista de frente o de canto tiene la misma morfología. Su relación
con la etiqueta sale de cómo se mide: se deduce del cociente de ejes `b/a` suponiendo un disco, así
que una elíptica redonda sale con inclinación baja, sin que eso diga nada de su orientación real.
Lo que la red verá es la **elongación** de la galaxia en la imagen.

El caso que preocupa son las **lenticulares (S0) vistas de canto**: son tempranas (`Ttype ≤ 0`),
pero alargadas como un disco. Si la red aprende "alargada = tardía", ahí se equivoca.

In [ ]:
# Familias a partir de Ttype: -5 = E, -2 = S0, 0 = S0a (las tres son tempranas), >= 1 = espirales e irregulares.
catalog["familia"] = pd.cut(catalog["Ttype"], [-np.inf, -4, 0, np.inf], labels=["E", "S0/S0a", "S/Irr"]).astype(str)
family_order = ["E", "S0/S0a", "S/Irr"]
fig, ax = plt.subplots(figsize=(8, 3.8))
bins_incl = np.linspace(0, 90, 19)
for family, linestyle in zip(family_order, ["-", "--", ":"]):
    group = catalog[catalog["familia"] == family]
    color = LABEL_COLORS[group["label"].mode()[0]]
    ax.hist(group["incl"], bins=bins_incl, density=True, histtype="step", linewidth=2,
            linestyle=linestyle, color=color, label=f"{family} (n = {len(group)})")
ax.set_xlabel("inclinación (grados)")
ax.set_ylabel("densidad")
ax.legend(frameon=False, title="familia de Type")
ax.set_title("Inclinación por familia morfológica", fontsize=10, loc="left")
plt.tight_layout()
plt.show()

pd.crosstab(catalog["familia"], catalog["label"])

In [ ]:
edge_on = catalog["incl"] >= 70
table = pd.DataFrame({
    "n": catalog.groupby("familia").size(),
    "de canto (incl >= 70°)": catalog[edge_on].groupby("familia").size(),
}).reindex(family_order).fillna(0).astype(int)
table["fraccion_de_canto"] = (table["de canto (incl >= 70°)"] / table["n"]).round(3)
table

- Las elípticas se concentran en inclinaciones bajas, y las espirales cubren todo el rango (como se
  espera de discos orientados al azar).
- Las **S0/S0a de canto** existen y son tempranas: son el grupo donde la elongación engaña. Al evaluar
  el modelo conviene reportar el error en ese subgrupo por separado.
- No se recomienda quitar la elongación (es parte de la morfología), sino **evaluar por bins de
  inclinación** y vigilar las S0 de canto y las espirales de frente sin brazos visibles.

## 7. Galaxias repetidas y vecinas dentro del encuadre

Una fuga que no es un atributo sino de **identidad**: si la misma galaxia (o la misma imagen) queda
en train y en test, el modelo la "reconoce". `galaxy_groups` junta en un mismo grupo:

- **Repeticiones:** la misma galaxia observada con dos plate-ifu distintos (separación < 3″).
- **Vecinas:** una galaxia que cae dentro del cutout de otra. Las dos imágenes comparten píxeles.

In [ ]:
groups = galaxy_groups(catalog)
catalog = catalog.join(groups)
multi = catalog[catalog["tam_grupo"] > 1]
print(f"entradas repetidas: {int(catalog['repetida'].sum())} ({int(catalog['repetida'].sum()) // 2} pares)")
print(f"entradas con una vecina en el encuadre: {int(catalog['vecina_en_encuadre'].sum())}")
print(f"grupos de más de una galaxia: {multi['grupo'].nunique()} (tamaño máximo {catalog['tam_grupo'].max()})")
same_label = multi.groupby("grupo")["label"].nunique().eq(1)
print(f"grupos con la misma etiqueta: {int(same_label.sum())} de {len(same_label)}")
multi.sort_values(["grupo", "name"])[["name", "grupo", "repetida", "vecina_en_encuadre", "Type", "label"]].head(12)

¿Cuántos grupos rompería un split al azar? Se compara una validación cruzada estratificada normal
con una que respeta los grupos (`StratifiedGroupKFold`):

In [ ]:
y_all = label_to_int(catalog["label"])
def broken_groups(splitter, **kwargs) -> int:
    fold = np.empty(len(catalog), dtype=int)
    for k, (_, test) in enumerate(splitter.split(np.zeros(len(catalog)), y_all, **kwargs)):
        fold[test] = k
    per_group = pd.Series(fold).groupby(catalog["grupo"].to_numpy()).nunique()
    return int((per_group > 1).sum())

print("grupos partidos, StratifiedKFold:     ",
      broken_groups(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)))
print("grupos partidos, StratifiedGroupKFold:",
      broken_groups(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED), groups=catalog["grupo"]))

Son pocas galaxias (< 1% del catálogo), así que el efecto en el AUC global será chico. Pero es
gratis evitarlo y, sobre todo, las **repeticiones son la misma imagen**: el split debe hacerse por
`grupo`, no por galaxia.

## 8. Primera mirada en la imagen: ¿delatan la clase las propiedades del `.fits`?

Con la muestra de 50 `.fits` se prueba si atributos **de la imagen** que no son morfología (ruido y
nivel del cielo por banda, zoom) predicen la etiqueta, con `predictability_test` (AUC con validación
cruzada repetida y p-valor por permutación). Con n = 50 la prueba solo detectaría señales fuertes:
sirve para montar el procedimiento, no para descartar atajos débiles.

In [ ]:
index = pd.read_csv(CUTOUTS_INDEX_CSV)
index = index[(index["format"] == "fits") & (index["layer"] == LEGACY_LAYER) & (index["bands"] == LEGACY_BANDS)]
sample50 = build_fits_photometry_table(index[["name", "label", "PETRO_TH90"]], LEGACY_LAYER, LEGACY_BANDS)

feature_sets = {
    "ruido del cielo (g, r, z)": [f"ruido_{b}" for b in LEGACY_BANDS],
    "nivel del cielo (g, r, z)": [f"cielo_{b}" for b in LEGACY_BANDS],
    "zoom (pixscale)": ["pixscale"],
    "control: colores g-r, r-z": ["g_r", "r_z"],
}
model = make_pipeline(StandardScaler(), LogisticRegression(C=SHORTCUT_LOGREG_C, max_iter=1000))
y50 = label_to_int(sample50["label"])
pd.DataFrame({
    name: predictability_test(model, sample50[columns].to_numpy(), y50)
    for name, columns in feature_sets.items()
}).T.round(3)

Los colores son el control positivo (sí son información física de la galaxia y deben separar). Las
propiedades del cielo y el zoom deberían quedar cerca de 0.5 con p alto; si alguna no, es candidata
a atajo y hay que medirla en una muestra más grande. La resolución (tamaño de la PSF) y la
profundidad por galaxia no están en el `.fits` y requieren el catálogo de Legacy Surveys.

## 9. Conclusiones y protocolo propuesto

| Atributo | ¿Predice la clase? | ¿Visible en la imagen? | Riesgo | Qué hacer |
|---|---|---|---|---|
| Redshift | Sí (AUC 0.67), pero **solo a través de la masa** (0.51 a masa igual) | Poco: el encuadre iguala el tamaño aparente | Bajo–moderado | Evaluar el modelo también con pesos `a redshift igual` |
| Región norte/sur | **No** (0.51) | Sí, mucho en `ls-dr10_grz` | Cambio de dominio, no atajo | Reportar AUC por región y cruzado norte→sur |
| Zoom / interpolación | No (0.52–0.55) | Sí (textura) | Bajo | Marcar zoom topado en el control de calidad |
| Inclinación / elongación | Sí (0.72), independiente de masa y redshift | Sí | Moderado (S0 de canto) | Evaluar por bins de inclinación y en S0 de canto |
| Repeticiones y vecinas | Fuga de identidad | Sí (misma imagen) | Bajo en volumen, seguro de evitar | Split por `grupo` (`StratifiedGroupKFold`) |
| Masa estelar | Sí (0.72) | Indirectamente (brillo, tamaño) | Correlato físico, no atajo | Comparar contra el baseline físico, no quitarla |

**Protocolo propuesto para los splits y la evaluación del modelo:**

1. **Split por grupo** (`galaxy_groups`), estratificado por etiqueta.
2. Reportar, además del AUC global:
   - AUC con pesos `a redshift igual` y `a masa igual` (`matching_weights` como `sample_weight`);
     si cae mucho, el modelo usa distancia o brillo en vez de forma;
   - AUC por región y entrenando en una región para evaluar en la otra (`cross_group_auc`);
   - error por bins de inclinación, con las S0 de canto aparte.
3. Repetir la sección 8 con el dataset completo y el tamaño de la PSF del catálogo de Legacy
   Surveys.